In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers

# -----------------------------
# 0) CONFIG
# -----------------------------
CSV_PATH = '../_filter/filter_Swap_log.csv'   # change if needed
SEQ_LEN = 30            # m - input sequence length
PRED_HORIZON = 1        # predict next step
BATCH_SIZE = 64
EPOCHS = 60
MODEL_DIR = 'model_v2'
os.makedirs(MODEL_DIR, exist_ok=True)

# categorical columns to embed (sequence-level)
CAT_SEQ = ['folio_index', 'mapping', 'lat_cluster',
           'PFN_Top_region','PFN_slice_4','PFN_slice_3','PFN_slice_2','PFN_slice_1','PFN_slice_0']
# meta categorical repeated per sequence
CAT_META = ['PID','Va_L4','Va_L3']
# numerical sequence columns
NUM_SEQ = ['Va_L2','Va_L1','PFN_delta','start_ns']

# -----------------------------
# 1) Load data
# -----------------------------
if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
else:
    # try to use df variable (if notebook already has it)
    try:
        df
    except NameError:
        raise FileNotFoundError(f"CSV not found at {CSV_PATH} and `df` not defined in namespace.")

# quick safety: required columns check
required_cols = set(CAT_SEQ + CAT_META + NUM_SEQ + ['PFN_Top_region','PFN_slice_4',
                                                     'PFN_slice_3','PFN_slice_2','PFN_slice_1','PFN_slice_0'])
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")


In [3]:

# -----------------------------
# 2) Basic preprocessing
# -----------------------------
# convert hex PFN column if present
if 'PFN_Hex' in df.columns:
    def hex_to_int(x):
        try:
            return int(str(x), 16)
        except:
            return np.nan
    df['PFN_val'] = df['PFN_Hex'].apply(hex_to_int).fillna(0).astype(np.int64)

# fillna simple
df[NUM_SEQ] = df[NUM_SEQ].fillna(0)
for c in CAT_SEQ + CAT_META:
    df[c] = df[c].fillna(0)

# compute latency cluster if not present
if 'lat_cluster' not in df.columns and 'latency_ns' in df.columns:
    from sklearn.cluster import KMeans
    kmeans = KMeans(n_clusters=4, random_state=42)
    df['lat_cluster'] = kmeans.fit_predict(df[['latency_ns']].fillna(0))
elif 'lat_cluster' not in df.columns:
    df['lat_cluster'] = 0

# ensure integer dtypes for categorical
for c in CAT_SEQ + CAT_META:
    df[c] = df[c].astype('int64')

# scale numerical sequence features
scalers = {c: StandardScaler() for c in NUM_SEQ}
for c in NUM_SEQ:
    df[c+'_scaled'] = scalers[c].fit_transform(df[[c]].astype(float))


In [4]:

# -----------------------------
# 3) Label encoding for categorical fields (map to 0..N-1)
#    Keep encoders for inference
# -----------------------------
encoders = {}
vocab_sizes = {}
for c in CAT_SEQ + CAT_META:
    le = LabelEncoder()
    df[c + '_enc'] = le.fit_transform(df[c].astype(str))
    encoders[c] = le
    vocab_sizes[c] = int(df[c + '_enc'].nunique())  # +1 safe

print('Vocab sizes:', vocab_sizes)


Vocab sizes: {'folio_index': 137, 'mapping': 168, 'lat_cluster': 5, 'PFN_Top_region': 2, 'PFN_slice_4': 14, 'PFN_slice_3': 16, 'PFN_slice_2': 16, 'PFN_slice_1': 16, 'PFN_slice_0': 16, 'PID': 24, 'Va_L4': 13, 'Va_L3': 17}


In [5]:

# -----------------------------
# 4) Build sequences (sliding window by PID + mapping)
#    We will group by (PID, mapping) to keep sequences process-local
# -----------------------------
SEQ_FEATURES = [f + '_scaled' for f in NUM_SEQ] + [c + '_enc' for c in CAT_SEQ]
META_FEATURES = [c + '_enc' for c in CAT_META]
print('SEQ_FEATURES:', SEQ_FEATURES)
print('META_FEATURES:', META_FEATURES)

# helper to build sequences
def build_sequences(df, seq_len=SEQ_LEN, group_cols=['PID','Va_L4','Va_L3']):
    X_seq = []
    X_meta = []
    Y = { 'top':[], 's4':[], 's3':[], 's2':[], 's1':[], 's0':[] }

    # sort by time inside group
    df_sorted = df.sort_values(['PID','Va_L4','Va_L3']).reset_index(drop=True)

    for _, g in df_sorted.groupby(group_cols):
        if len(g) <= seq_len: 
            continue
        arr_num = g[SEQ_FEATURES].values
        arr_meta = g[META_FEATURES].values
        # targets: next-step slice labels (already integer-coded in PFN_slice_x columns)
        topo = g['PFN_Top_region'].values
        s4 = g['PFN_slice_4'].values
        s3 = g['PFN_slice_3'].values
        s2 = g['PFN_slice_2'].values
        s1 = g['PFN_slice_1'].values
        s0 = g['PFN_slice_0'].values

        # sliding windows
        for i in range(len(g) - seq_len - (PRED_HORIZON-1)):
            x_seq = arr_num[i:i+seq_len]
            # take meta from the last timestep of the input window (repeatable features)
            x_meta = arr_meta[i+seq_len-1]
            y_idx = i+seq_len  # next-step index
            X_seq.append(x_seq)
            X_meta.append(x_meta)
            Y['top'].append(topo[y_idx])
            Y['s4'].append(s4[y_idx])
            Y['s3'].append(s3[y_idx])
            Y['s2'].append(s2[y_idx])
            Y['s1'].append(s1[y_idx])
            Y['s0'].append(s0[y_idx])

    # convert to arrays
    X_seq = np.asarray(X_seq, dtype=np.float32)
    X_meta = np.asarray(X_meta, dtype=np.int32)
    for k in Y:
        Y[k] = np.asarray(Y[k], dtype=np.int32)

    return X_seq, X_meta, Y

X_seq, X_meta, Y = build_sequences(df)
print('Built sequences ->', X_seq.shape, X_meta.shape, {k:v.shape for k,v in Y.items()})


SEQ_FEATURES: ['Va_L2_scaled', 'Va_L1_scaled', 'PFN_delta_scaled', 'start_ns_scaled', 'folio_index_enc', 'mapping_enc', 'lat_cluster_enc', 'PFN_Top_region_enc', 'PFN_slice_4_enc', 'PFN_slice_3_enc', 'PFN_slice_2_enc', 'PFN_slice_1_enc', 'PFN_slice_0_enc']
META_FEATURES: ['PID_enc', 'Va_L4_enc', 'Va_L3_enc']
Built sequences -> (2578, 30, 13) (2578, 3) {'top': (2578,), 's4': (2578,), 's3': (2578,), 's2': (2578,), 's1': (2578,), 's0': (2578,)}


In [6]:

# -----------------------------
# 5) Train / val split
# -----------------------------
X_seq_train, X_seq_val, X_meta_train, X_meta_val = train_test_split(
    X_seq, X_meta, test_size=0.2, random_state=42)

Y_train = {k: v for k,v in Y.items()}
Y_val = {k: v for k,v in Y.items()}  # placeholder: we'll re-slice below
# We must select same train/val indices; using sklearn split above yields indices we don't have
# Easiest: compute indices with permutation
n = X_seq.shape[0]
idx = np.arange(n)
train_idx, val_idx = train_test_split(idx, test_size=0.2, random_state=42)

X_seq_train = X_seq[train_idx]
X_seq_val   = X_seq[val_idx]
X_meta_train = X_meta[train_idx]
X_meta_val   = X_meta[val_idx]

Y_train = { k: v[train_idx] for k,v in Y.items() }
Y_val   = { k: v[val_idx]   for k,v in Y.items() }

print('Train samples:', X_seq_train.shape[0], 'Val samples:', X_seq_val.shape[0])


Train samples: 2062 Val samples: 516


In [7]:

# -----------------------------
# 6) Build model
# -----------------------------
# Inputs
seq_input = layers.Input(shape=(SEQ_LEN, len(SEQ_FEATURES)), name='seq_input')
meta_input = layers.Input(shape=(len(META_FEATURES),), dtype='int32', name='meta_input')

# Embeddings for the categorical parts inside the sequence features
# We know first 4 features are scaled numerics; then the categorical enc indexes follow
num_num = len(NUM_SEQ)
cat_seq_names = [c for c in CAT_SEQ]
cat_meta_names = [c for c in CAT_META]

# split numeric and categorical from seq_input
num_part = layers.Lambda(lambda x: x[:, :, :num_num], name='num_part')(seq_input)
cat_part = layers.Lambda(lambda x: x[:, :, num_num:], name='cat_part')(seq_input)

# cat_part has encoded integers but currently are floats in seq_input - cast
cat_part_int = layers.Lambda(lambda x: tf.cast(tf.round(x), tf.int32), name='cat_part_int')(cat_part)

# create embedding layers for each categorical slice inside sequence
embedded_seq_parts = []
for i, name in enumerate(cat_seq_names):
    vocab = vocab_sizes[name]
    emb_dim = min(32, max(4, int(vocab**0.25*8)))
    # extract the i-th categorical column along last axis
    col = layers.Lambda(lambda x, idx=i: x[:, :, idx], name=f'seq_{name}_col')(cat_part_int)
    emb = layers.Embedding(input_dim=vocab, output_dim=emb_dim, mask_zero=False, name=f'emb_seq_{name}')(col)
    embedded_seq_parts.append(emb)

# concat numeric + embedded categorical
num_part_proj = layers.TimeDistributed(layers.Dense(16, activation='relu'))(num_part)
seq_merged = layers.Concatenate(axis=-1)([num_part_proj] + embedded_seq_parts)

# GRU stack
x = layers.GRU(256, return_sequences=True, name='gru1')(seq_merged)
x = layers.Dropout(0.2)(x)
x = layers.GRU(128, return_sequences=True, name='gru2')(x)
x = layers.Dropout(0.2)(x)
x = layers.GRU(128, return_sequences=True, name='gru3')(x)
x = layers.Dropout(0.2)(x)
x = layers.GRU(64, return_sequences=True, name='gru5')(x)
x = layers.Dropout(0.2)(x)
x = layers.GRU(32, return_sequences=False, name='gru6')(x)
x = layers.Dropout(0.2)(x)

# META embeddings (PID, Va_L4, Va_L3)
meta_embs = []
for j, name in enumerate(cat_meta_names):
    vocab = vocab_sizes[name]
    emb_dim = min(32, max(4, int(vocab**0.25*8)))
    e = layers.Embedding(input_dim=vocab, output_dim=emb_dim, name=f'emb_meta_{name}')(meta_input[:,j])
    meta_embs.append(e)

meta_concat = layers.Concatenate()(meta_embs)
meta_proj = layers.Dense(64, activation='relu')(meta_concat)

# combine
combined = layers.Concatenate()([x, meta_proj])
shared = layers.Dense(256, activation='relu')(combined)
shared = layers.Dropout(0.2)(shared)

# multi-head outputs
def make_head(name, classes):
    h = layers.Dense(128, activation='relu')(shared)
    return layers.Dense(classes, activation='softmax', name=name)(h)

out_top = make_head('top_out', vocab_sizes['PFN_Top_region'])
out_s4  = make_head('s4_out',  vocab_sizes['PFN_slice_4'])
out_s3  = make_head('s3_out',  vocab_sizes['PFN_slice_3'])
out_s2  = make_head('s2_out',  vocab_sizes['PFN_slice_2'])
out_s1  = make_head('s1_out',  vocab_sizes['PFN_slice_1'])
out_s0  = make_head('s0_out',  vocab_sizes['PFN_slice_0'])

model = models.Model(inputs=[seq_input, meta_input], outputs=[out_top,out_s4,out_s3,out_s2,out_s1,out_s0])

# compile with per-output losses and metrics
losses = {
    'top_out': 'sparse_categorical_crossentropy',
    's4_out': 'sparse_categorical_crossentropy',
    's3_out': 'sparse_categorical_crossentropy',
    's2_out': 'sparse_categorical_crossentropy',
    's1_out': 'sparse_categorical_crossentropy',
    's0_out': 'sparse_categorical_crossentropy',
}
metrics = {k: 'accuracy' for k in losses.keys()}

optimizer = optimizers.Adam(learning_rate=1e-4)
model.compile(optimizer=optimizer, loss=losses, metrics=metrics)
# model.summary()


In [8]:

# -----------------------------
# 7) Prepare dataset objects for Keras
# -----------------------------
train_y = {
    'top_out': Y_train['top'], 's4_out': Y_train['s4'], 's3_out': Y_train['s3'],
    's2_out': Y_train['s2'], 's1_out': Y_train['s1'], 's0_out': Y_train['s0']
}
val_y = {
    'top_out': Y_val['top'], 's4_out': Y_val['s4'], 's3_out': Y_val['s3'],
    's2_out': Y_val['s2'], 's1_out': Y_val['s1'], 's0_out': Y_val['s0']
}

# convert meta_input from shape (batch, N_meta) to integers for embedding usage (Keras expects 2D ints)
# Already X_meta_train is int32

# callbacks
cb = [
    callbacks.EarlyStopping(monitor='val_loss', patience=40, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=40),
    callbacks.ModelCheckpoint(os.path.join(MODEL_DIR,'best.keras'), save_best_only=True)
]


In [9]:

# -----------------------------
# 8) Train
# -----------------------------
history = model.fit(
    {'seq_input': X_seq_train, 'meta_input': X_meta_train},
    train_y,
    validation_data=({'seq_input': X_seq_val, 'meta_input': X_meta_val}, val_y),
    epochs=20, batch_size=BATCH_SIZE, callbacks=cb
)

# save model and encoders
model.save(os.path.join(MODEL_DIR,'final_model.keras'))
import pickle
with open(os.path.join(MODEL_DIR,'encoders.pkl'),'wb') as f:
    pickle.dump({'encoders': encoders, 'scalers': scalers, 'vocab_sizes': vocab_sizes}, f)


Epoch 1/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 23s 235ms/step - loss: 14.3953 - s0_out_accuracy: 0.0679 - s1_out_accuracy: 0.0620 - s2_out_accuracy: 0.0755 - s3_out_accuracy: 0.0566 - s4_out_accuracy: 0.1650 - top_out_accuracy: 0.5362 - val_loss: 14.2230 - val_s0_out_accuracy: 0.0426 - val_s1_out_accuracy: 0.0717 - val_s2_out_accuracy: 0.0504 - val_s3_out_accuracy: 0.0698 - val_s4_out_accuracy: 0.3333 - val_top_out_accuracy: 0.6473 - learning_rate: 1.0000e-04
Epoch 2/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 6s 177ms/step - loss: 14.0865 - s0_out_accuracy: 0.0736 - s1_out_accuracy: 0.0666 - s2_out_accuracy: 0.0612 - s3_out_accuracy: 0.0683 - s4_out_accuracy: 0.2908 - top_out_accuracy: 0.6529 - val_loss: 13.8096 - val_s0_out_accuracy: 0.0678 - val_s1_out_accuracy: 0.0698 - val_s2_out_accuracy: 0.0368 - val_s3_out_accuracy: 0.0601 - val_s4_out_accuracy: 0.3062 - val_top_out_accuracy: 0.6473 - learning_rate: 1.0000e-04
Epoch 3/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 6s 176ms/step - loss: 13.7899 - s0_out_accuracy: 0.07

In [10]:

# -----------------------------
# 9) Evaluation: report per-head accuracy on validation
# -----------------------------
val_pred = model.predict({'seq_input': X_seq_val, 'meta_input': X_meta_val})
for name, pred, true in zip(['top','s4','s3','s2','s1','s0'], val_pred, [Y_val['top'],Y_val['s4'],Y_val['s3'],Y_val['s2'],Y_val['s1'],Y_val['s0']]):
    pred_cls = np.argmax(pred, axis=-1)
    acc = (pred_cls == true).mean()
    print(f"Val accuracy {name}: {acc:.4f}")

# -----------------------------
# 10) PFN reconstruction helper
# -----------------------------
# Reconstruct integer PFN from predicted slices (approx). This needs your bit layout.
# Example combining: PFN_val = (top<<20) | (s4<<16) | (s3<<12) | (s2<<8) | (s1<<4) | s0

def reconstruct_pfn(top, s4, s3, s2, s1, s0):
    return (top.astype(np.int64) << 20) | (s4.astype(np.int64) << 16) | (s3.astype(np.int64) << 12) | \
           (s2.astype(np.int64) << 8) | (s1.astype(np.int64) << 4) | (s0.astype(np.int64))

recon = reconstruct_pfn(np.argmax(val_pred[0],-1), np.argmax(val_pred[1],-1), np.argmax(val_pred[2],-1),
                        np.argmax(val_pred[3],-1), np.argmax(val_pred[4],-1), np.argmax(val_pred[5],-1))
mae = np.mean(np.abs(recon - df['PFN_val'].values[val_idx]))
print('Reconstructed PFN MAE (val set):', mae)


17/17 ━━━━━━━━━━━━━━━━━━━━ 3s 98ms/step
Val accuracy top: 0.8527
Val accuracy s4: 0.4244
Val accuracy s3: 0.1260
Val accuracy s2: 0.1279
Val accuracy s1: 0.0911
Val accuracy s0: 0.1085
Reconstructed PFN MAE (val set): 425516.08914728684
